In [1]:
%pip install -U -q pip setuptools wheel setuptools-scm
%pip uninstall -y -q diffusers
%pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 seqeval==1.2.2
%pip install -q optimum optimum-onnx onnx==1.16.2 onnxruntime==1.19.2 onnxscript
print("ok")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.1/109.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done

In [2]:
ENTITY_TYPES = ["ITEM", "QTY", "UNIT", "VARIANT", "ANAPHORIC"]
LABELS = ["O"] + [f"{p}-{e}" for e in ENTITY_TYPES for p in ("B", "I")]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(f"{len(LABELS)} labels:", LABELS)

11 labels: ['O', 'B-ITEM', 'I-ITEM', 'B-QTY', 'I-QTY', 'B-UNIT', 'I-UNIT', 'B-VARIANT', 'I-VARIANT', 'B-ANAPHORIC', 'I-ANAPHORIC']


In [3]:
import json, sys, importlib

DATA = "/kaggle/input/datasets/sebastianabe/nyatet-order-train"
B5 = "/kaggle/input/datasets/sebastianabe/nyatet-order-train-b5"

sys.path.insert(0, DATA)
import generate_data
importlib.reload(generate_data)

train_rows = generate_data.generate_dataset(n_orders=8000)
generate_data.validate(train_rows)

eval_rows = json.load(open(f"{B5}/eval_annotated.json", encoding="utf-8"))

print(f"train {len(train_rows)}  |  eval {len(eval_rows)} (real, held out)")

for name, rows in (("train", train_rows), ("eval", eval_rows)):
    for r in rows:
        for s in r["spans"]:
            frag = r["text"][s["start"]:s["end"]]
            assert frag and frag == frag.strip(), f"{name}: bad span {frag!r} in {r['text']!r}"
print("offset check passed")

train 10800  |  eval 81 (real, held out)
offset check passed


In [4]:
from collections import Counter
print("train spans:", dict(Counter(s["type"] for r in train_rows for s in r["spans"])))
print("eval  spans:", dict(Counter(s["type"] for r in eval_rows for s in r["spans"])))
print()
for r in eval_rows[:4]:
    print(repr(r["text"]))
    for s in r["spans"]:
        print(f'    {s["type"]:<10} {r["text"][s["start"]:s["end"]]!r}')

train spans: {'ITEM': 3939, 'ANAPHORIC': 345, 'QTY': 9417, 'UNIT': 5984, 'VARIANT': 4274}
eval  spans: {'ITEM': 21, 'QTY': 35, 'UNIT': 21, 'VARIANT': 19, 'ANAPHORIC': 2}

'Pesan hari ni risol 20 adakah Bu, kemarin Ulun kewarung Pian kd bejualan☺️'
    ITEM       'risol'
    QTY        '20'
'Tolong dibalas chatnya soalnya klo kd  jualan Ulun kd mendatangi soalnya jauh jaraknya bu🙏🏻'
'Esok tinggali 20 biji ya, jam 08.15 diambil'
    QTY        '20'
    UNIT       'biji'
'Inggih digoreng spt biasa🙏🏻'
    VARIANT    'digoreng'
    ANAPHORIC  'spt biasa'


In [5]:
from transformers import BertTokenizerFast

MODEL_NAME = "indobenchmark/indobert-lite-base-p2"
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

def encode(row):
    enc = tokenizer(row["text"], return_offsets_mapping=True,
                    truncation=True, max_length=96)
    offsets = enc.pop("offset_mapping")
    tags = [None if a == b else "O" for a, b in offsets]
    for span in row["spans"]:
        entered = False
        for i, (a, b) in enumerate(offsets):
            if a == b:
                continue
            if a >= span["start"] and b <= span["end"]:
                tags[i] = f'{"B" if not entered else "I"}-{span["type"]}'
                entered = True
    enc["labels"] = [-100 if t is None else LABEL2ID[t] for t in tags]
    return enc

r = eval_rows[2]
e = encode(r)
print(repr(r["text"]), "\n")
for t, l in zip(tokenizer.convert_ids_to_tokens(e["input_ids"]), e["labels"]):
    print(f"{t:<18} {ID2LABEL[l] if l != -100 else '(ignored)'}")

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizerFast'.


'Esok tinggali 20 biji ya, jam 08.15 diambil' 

[CLS]              (ignored)
esok               O
tinggal            O
##i                O
20                 B-QTY
biji               B-UNIT
ya                 O
,                  O
jam                O
08                 O
.                  O
15                 O
diambil            O
[SEP]              (ignored)


In [6]:
from datasets import Dataset

train_ds = Dataset.from_list([encode(r) for r in train_rows]).train_test_split(test_size=0.05, seed=7)
eval_ds  = Dataset.from_list([encode(r) for r in eval_rows])
print(train_ds); print(eval_ds)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10260
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 540
    })
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 81
})


In [7]:
import numpy as np, torch, random
from seqeval.metrics import f1_score, classification_report
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)

SEED = 42
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    tp = [[ID2LABEL[a] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    tl = [[ID2LABEL[b] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    return {"f1": f1_score(tl, tp)}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./checkpoints/order", num_train_epochs=5,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=5e-5, eval_strategy="epoch", save_strategy="no",
        logging_steps=50, report_to="none", fp16=True, seed=SEED),
    train_dataset=train_ds["train"], eval_dataset=train_ds["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/46.7M [00:00<?, ?B/s]

Some weights of AlbertForTokenClassification were not initialized from the model checkpoint at indobenchmark/indobert-lite-base-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,0.000000,0.000018,1.000000
2,0.000000,0.000000,1.000000
3,0.000000,0.000000,1.000000
4,0.000000,0.000000,1.000000
5,0.000000,0.000000,1.000000


TrainOutput(global_step=805, training_loss=0.014074855937264978, metrics={'train_runtime': 107.6113, 'train_samples_per_second': 476.716, 'train_steps_per_second': 7.481, 'total_flos': 44955484037856.0, 'train_loss': 0.014074855937264978, 'epoch': 5.0})

In [8]:
def report(dataset, title):
    preds, labels, _ = trainer.predict(dataset)
    preds = np.argmax(preds, axis=2)
    tp = [[ID2LABEL[a] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    tl = [[ID2LABEL[b] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    print(f"=== {title} ===\nF1 {f1_score(tl, tp):.4f}\n")
    print(classification_report(tl, tp, digits=3))
    return tl, tp

_ = report(train_ds["test"], "SYNTHETIC held-out — diagnostic only")

=== SYNTHETIC held-out — diagnostic only ===
F1 1.0000

              precision    recall  f1-score   support

   ANAPHORIC      1.000     1.000     1.000        19
        ITEM      1.000     1.000     1.000       167
         QTY      1.000     1.000     1.000       461
        UNIT      1.000     1.000     1.000       283
     VARIANT      1.000     1.000     1.000       213

   micro avg      1.000     1.000     1.000      1143
   macro avg      1.000     1.000     1.000      1143
weighted avg      1.000     1.000     1.000      1143



In [9]:
tl, tp = report(eval_ds, f"REAL held-out (n={len(eval_rows)})")

=== REAL held-out (n=81) ===
F1 0.8365

              precision    recall  f1-score   support

   ANAPHORIC      0.500     0.500     0.500         2
        ITEM      0.947     0.857     0.900        21
         QTY      0.895     0.971     0.932        35
        UNIT      0.909     0.952     0.930        21
     VARIANT      0.483     0.737     0.583        19

   micro avg      0.791     0.888     0.837        98
   macro avg      0.747     0.804     0.769        98
weighted avg      0.821     0.888     0.848        98



In [10]:
for row, gold, pred in zip(eval_rows, tl, tp):
    if gold != pred:
        print(repr(row["text"]))
        enc = tokenizer(row["text"], return_offsets_mapping=True, truncation=True, max_length=96)
        toks = [t for t, (a, b) in zip(tokenizer.convert_ids_to_tokens(enc["input_ids"]),
                                       enc["offset_mapping"]) if a != b]
        for t, g, p in zip(toks, gold, pred):
            if g != p:
                print(f"    {t:<15} gold {g:<12} pred {p}")
        print()

'Inggih digoreng spt biasa🙏🏻'
    spt             gold B-ANAPHORIC  pred O

'Bu tinggalijam 7.30 ya bu'
    ##ij            gold O            pred I-VARIANT
    ##am            gold O            pred I-VARIANT
    7               gold O            pred B-QTY
    30              gold O            pred B-QTY

'Besok jualanlah risol'
    ris             gold B-ITEM       pred O
    ##ol            gold I-ITEM       pred O

'Yg mentah 10'
    yg              gold O            pred B-VARIANT

'Iya jd 55 ribu kalo duitnya'
    55              gold O            pred B-QTY
    ribu            gold O            pred B-UNIT

'Besok adalah risol tp yg mentahnya aja 10 buting'
    tp              gold O            pred B-VARIANT
    yg              gold O            pred B-VARIANT
    mentah          gold B-VARIANT    pred I-VARIANT

'Jam setengah 9 diambil bisa'
    9               gold O            pred B-QTY

'Oke yg mentahlah besok 10'
    yg              gold O            pred B-VARIANT
    1

In [11]:
trainer.save_model("./checkpoints/order"); tokenizer.save_pretrained("./checkpoints/order")

from optimum.onnxruntime import ORTModelForTokenClassification
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
from pathlib import Path
import shutil

m = ORTModelForTokenClassification.from_pretrained("./checkpoints/order", export=True)
m.save_pretrained("./onnx_fp32"); tokenizer.save_pretrained("./onnx_fp32")

quant_pre_process(
    input_model_path="onnx_fp32/model.onnx",
    output_model_path="onnx_fp32/model_preprocessed.onnx",
    skip_symbolic_shape=True)

for name, ops in (("onnx_int8_v2", ["MatMul"]), ("onnx_int8_v3", ["MatMul", "Gather"])):
    Path(name).mkdir(parents=True, exist_ok=True)
    quantize_dynamic("onnx_fp32/model_preprocessed.onnx", f"{name}/model.onnx",
                     weight_type=QuantType.QInt8,
                     op_types_to_quantize=ops,
                     extra_options={"MatMulConstBOnly": False})
    tokenizer.save_pretrained(name)
    shutil.copy("./checkpoints/order/config.json", f"{name}/config.json")

for d in ("onnx_fp32", "onnx_int8_v2", "onnx_int8_v3"):
    for f in sorted(Path(d).glob("*.onnx")):
        print(f"{str(f):<42} {f.stat().st_size/1024**2:6.2f} MB")

onnx_fp32/model.onnx                        42.63 MB
onnx_fp32/model_preprocessed.onnx           42.56 MB
onnx_int8_v2/model.onnx                     22.17 MB
onnx_int8_v3/model.onnx                     11.00 MB


In [12]:
import time, onnxruntime as ort
from pathlib import Path

def eval_onnx(path, report=False):
    sess = ort.InferenceSession(path)
    names = [i.name for i in sess.get_inputs()]
    tp, tl = [], []
    for row in eval_rows:
        enc = tokenizer(row["text"], truncation=True, max_length=96, return_tensors="np")
        feed = {n: enc[n].astype(np.int64) for n in names}
        pred = sess.run(None, feed)[0][0].argmax(-1)
        gold = encode(row)["labels"]
        for p, g in zip(pred, gold):
            if g != -100:
                tp.append(ID2LABEL[int(p)]); tl.append(ID2LABEL[int(g)])
    if report:
        print(classification_report([tl], [tp], digits=3))
    return f1_score([tl], [tp])

def bench(path, n=100):
    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    s = ort.InferenceSession(path, so)
    enc = tokenizer("bu risol mentah 20 biji, jam 7 pagi diambil", return_tensors="np")
    feed = {i.name: enc[i.name].astype(np.int64) for i in s.get_inputs()}
    for _ in range(20): s.run(None, feed)
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); s.run(None, feed); ts.append((time.perf_counter()-t0)*1000)
    ts.sort()
    return ts[n//2], ts[int(n*0.95)-1]

MODELS = {"fp32": "onnx_fp32/model.onnx",
          "v2 (MatMul only)": "onnx_int8_v2/model.onnx",
          "v3 (MatMul+Gather)": "onnx_int8_v3/model.onnx"}

print(f"{'variant':<22} {'F1':>7} {'size MB':>9} {'median':>8} {'p95':>7}")
for name, path in MODELS.items():
    if not Path(path).exists():
        print(f"{name:<22}   -- missing --"); continue
    med, p95 = bench(path)
    print(f"{name:<22} {eval_onnx(path):>7.4f} {Path(path).stat().st_size/1024**2:>9.2f} "
          f"{med:>7.1f}ms {p95:>6.1f}ms")

print("\n--- per-class, v2 ---")
eval_onnx("onnx_int8_v2/model.onnx", report=True)

variant                     F1   size MB   median     p95
fp32                    0.8365     42.63    41.2ms   43.0ms
v2 (MatMul only)        0.8502     22.17    20.6ms   21.5ms
v3 (MatMul+Gather)      0.8476     11.00    20.4ms   21.8ms

--- per-class, v2 ---
              precision    recall  f1-score   support

   ANAPHORIC      1.000     0.500     0.667         2
        ITEM      0.947     0.857     0.900        21
         QTY      0.895     0.971     0.932        35
        UNIT      0.909     0.952     0.930        21
     VARIANT      0.517     0.789     0.625        19

   micro avg      0.807     0.898     0.850        98
   macro avg      0.854     0.814     0.811        98
weighted avg      0.838     0.898     0.860        98



np.float64(0.8502415458937198)

In [13]:
import zipfile
from pathlib import Path

SKIP = {"model_preprocessed.onnx"}

for d in ["onnx_fp32", "onnx_int8_v2", "onnx_int8_v3"]:
    p = Path(d)
    if not p.exists():
        print(f"skip {d} (missing)")
        continue
    out = Path(f"/kaggle/working/{d}.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(p.rglob("*")):
            if f.is_file() and f.name not in SKIP:
                z.write(f, f.name)
    print(f"{out.name:<24} {out.stat().st_size/1024**2:6.2f} MB")

onnx_fp32.zip             39.63 MB
onnx_int8_v2.zip          19.10 MB
onnx_int8_v3.zip           8.42 MB


In [14]:
import onnxruntime as ort, numpy as np

sess = ort.InferenceSession("onnx_int8_v3/model.onnx")
names = [i.name for i in sess.get_inputs()]

fp_msgs, fp_spans = 0, 0
for row in eval_rows:
    if row["spans"]: continue
    enc = tokenizer(row["text"], truncation=True, max_length=96, return_tensors="np")
    feed = {n: enc[n].astype(np.int64) for n in names}
    pred = sess.run(None, feed)[0][0].argmax(-1)
    gold = encode(row)["labels"]
    hits = [ID2LABEL[int(p)] for p, g in zip(pred, gold) if g != -100 and ID2LABEL[int(p)] != "O"]
    if hits:
        fp_msgs += 1
        fp_spans += len(hits)
        print(f"  {row['text'][:55]:<55} -> {hits}")

n_neg = sum(1 for r in eval_rows if not r["spans"])
print(f"\n{fp_msgs}/{n_neg} negatives got a spurious span ({fp_msgs/n_neg*100:.0f}%)")
print(f"{fp_spans} spurious tokens total")

  Bu tinggalijam 7.30 ya bu                               -> ['I-VARIANT', 'B-QTY', 'B-QTY']
  Iya jd 55 ribu kalo duitnya                             -> ['B-QTY', 'B-QTY']
  Jam setengah 9 diambil bisa                             -> ['B-QTY']
  Yg pian kawa pastikan ibu ambil yg jam 6                -> ['B-ANAPHORIC', 'I-ANAPHORIC']
  Yg nia olah ukuran berapa                               -> ['I-VARIANT', 'I-VARIANT']

5/41 negatives got a spurious span (12%)
10 spurious tokens total
